In [1]:
from pathlib import Path
import json
import pandas as pd

# ---- Load all JSONL files in current folder ----
jsonl_files = sorted(Path(".").glob("*.jsonl"))
if not jsonl_files:
    raise FileNotFoundError("No .jsonl files found in current folder.")

def pick_task_id(rec):
    for k in ["task_id", "task", "id", "problem_id", "question_id", "name"]:
        if k in rec and rec[k] is not None:
            return str(rec[k])
    return None

def pick_solved(rec):
    # Preferred explicit fields
    for k in ["solved", "is_solved", "correct", "is_correct", "passed", "success"]:
        if k in rec:
            v = rec[k]
            if isinstance(v, bool):
                return v
            if isinstance(v, (int, float)):
                return v > 0
            if isinstance(v, str):
                s = v.strip().lower()
                if s in {"true", "t", "yes", "y", "1", "pass", "passed", "correct", "solved", "ok", "success"}:
                    return True
                if s in {"false", "f", "no", "n", "0", "fail", "failed", "incorrect", "unsolved", "error"}:
                    return False

    # Fallback status-style fields
    for k in ["status", "result", "outcome", "verdict"]:
        if k in rec and rec[k] is not None:
            s = str(rec[k]).strip().lower()
            if s in {"pass", "passed", "correct", "solved", "ok", "success"}:
                return True
            if s in {"fail", "failed", "incorrect", "unsolved", "error"}:
                return False
    return None

rows = []
for fp in jsonl_files:
    model = fp.stem  # model name from file name
    with fp.open("r", encoding="utf-8") as f:
        for ln, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            task_id = pick_task_id(rec)
            solved = pick_solved(rec)
            if task_id is None or solved is None:
                continue
            rows.append({"model": model, "task_id": task_id, "solved": bool(solved)})

dataset = pd.DataFrame(rows)
if dataset.empty:
    raise ValueError("Parsed dataset is empty. Check JSONL schema or field names.")

# If duplicates exist for same model/task, keep max (True if any run solved)
dataset = (
    dataset.groupby(["task_id", "model"], as_index=False)["solved"]
    .max()
)

# ---- Analysis ----
task_model = dataset.pivot(index="task_id", columns="model", values="solved").fillna(False).astype(bool)
n_models = task_model.shape[1]

solved_by_n = task_model.sum(axis=1).astype(int)  # per task: how many models solved
distribution = solved_by_n.value_counts().reindex(range(0, n_models + 1), fill_value=0).sort_index()

tasks_solved_by_all = solved_by_n[solved_by_n == n_models].index.tolist()
tasks_solved_by_none = solved_by_n[solved_by_n == 0].index.tolist()
tasks_by_k = {k: solved_by_n[solved_by_n == k].index.tolist() for k in range(0, n_models + 1)}

print(f"Loaded files: {len(jsonl_files)}")
print(f"Models detected: {n_models}")
print(f"Tasks detected: {task_model.shape[0]}")
print("\n# tasks solved by k models")
print(distribution.to_string())

print("\nTasks solved by ALL models:")
print(tasks_solved_by_all)

print("\nTasks solved by 0 models:")
print(tasks_solved_by_none)

# Optional tables to inspect in notebook
summary_distribution = distribution.rename("num_tasks").reset_index().rename(columns={"index": "k_models_solved"})
summary_per_task = solved_by_n.rename("k_models_solved").reset_index()

Loaded files: 7
Models detected: 7
Tasks detected: 65

# tasks solved by k models
0    19
1    12
2    13
3    12
4     4
5     4
6     1
7     0

Tasks solved by ALL models:
[]

Tasks solved by 0 models:
['vietjack_math_v4_0004', 'vietjack_math_v4_0008', 'vietjack_math_v4_0010', 'vietjack_math_v4_0012', 'vietjack_math_v4_0015', 'vietjack_math_v4_0019', 'vietjack_math_v4_0020', 'vietjack_math_v4_0024', 'vietjack_math_v4_0027', 'vietjack_math_v4_0036', 'vietjack_math_v4_0040', 'vietjack_math_v4_0043', 'vietjack_math_v4_0044', 'vietjack_math_v4_0045', 'vietjack_math_v4_0047', 'vietjack_math_v4_0048', 'vietjack_math_v4_0052', 'vietjack_math_v4_0054', 'vietjack_math_v4_0063']


In [2]:
# Print tasks solved by 0 models and try to show their expected result (if present in JSONL records)

expected_keys = [
    "expected", "expected_result", "ground_truth", "answer", "target",
    "label", "reference", "gold", "gold_answer", "canonical_solution"
]

none_set = set(tasks_solved_by_none)
task_expected = {}

for fp in jsonl_files:
    with fp.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue

            tid = pick_task_id(rec)
            if tid not in none_set or tid in task_expected:
                continue

            exp_val = None
            for k in expected_keys:
                if k in rec and rec[k] is not None:
                    exp_val = rec[k]
                    break

            task_expected[tid] = exp_val

for i, tid in enumerate(tasks_solved_by_none, 1):
    exp_val = task_expected.get(tid, None)
    print(f"{i}. task_id: {tid}")
    print(f"   expected: {exp_val if exp_val is not None else '[not found in JSONL fields]'}")

1. task_id: vietjack_math_v4_0004
   expected: [not found in JSONL fields]
2. task_id: vietjack_math_v4_0008
   expected: [not found in JSONL fields]
3. task_id: vietjack_math_v4_0010
   expected: [not found in JSONL fields]
4. task_id: vietjack_math_v4_0012
   expected: [not found in JSONL fields]
5. task_id: vietjack_math_v4_0015
   expected: [not found in JSONL fields]
6. task_id: vietjack_math_v4_0019
   expected: [not found in JSONL fields]
7. task_id: vietjack_math_v4_0020
   expected: [not found in JSONL fields]
8. task_id: vietjack_math_v4_0024
   expected: [not found in JSONL fields]
9. task_id: vietjack_math_v4_0027
   expected: [not found in JSONL fields]
10. task_id: vietjack_math_v4_0036
   expected: [not found in JSONL fields]
11. task_id: vietjack_math_v4_0040
   expected: [not found in JSONL fields]
12. task_id: vietjack_math_v4_0043
   expected: [not found in JSONL fields]
13. task_id: vietjack_math_v4_0044
   expected: [not found in JSONL fields]
14. task_id: vietjack